# Gradient Calibration Tutorial

Calibrating regularization weights by matching gradient magnitudes.

**Problem:** Regularization losses (voltage, rate) must be balanced against task loss (CE). If they're too strong, they dominate training. If too weak, they have no effect.

**Solution:** Use gradient magnitude matching to set weights such that reg gradients are ~10% of task gradients.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from btorch.models import environ, functional
from btorch.models.init import uniform_v_

from src.model import SingleLayerGLIFRSNN
from src.loss import CombinedLoss, LossConfig
from src.calibration_grad import calibrate_regularization_weights, quick_calibration_check

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Create Model and Dummy Data

In [ ]:
model = SingleLayerGLIFRSNN(
    input_dim=80,
    output_dim=10,
    n_neuron=128,
    n_adapt=64,
).to(device)

# Initialize
batch_size = 32
T = 100
functional.init_net_state(model.rnn, batch_size=batch_size, device=device)
uniform_v_(model.neuron, set_reset_value=True)

# Dummy dataloader
from torch.utils.data import TensorDataset, DataLoader

dummy_data = torch.randn(100, T, 80)
dummy_labels = torch.randint(0, 10, (100,))
dataset = TensorDataset(dummy_data, dummy_labels)
loader = DataLoader(dataset, batch_size=32)

print("Model and data ready")

## 2. Quick Calibration Check

First, check current loss component ratios. Target: reg losses ~10-20% of CE loss.

In [ ]:
# Initial loss config
loss_config = LossConfig(
    ce_weight=1.0,
    voltage_weight=0.01,
    rate_weight=0.1,
)
loss_fn = CombinedLoss(loss_config, dt=1.0)

# Quick check
recommendations = quick_calibration_check(model, loader, loss_fn, device, num_batches=3)

print("Current loss values:")
print(f"  CE loss: {recommendations['ce_loss']:.4f}")
print(f"  Voltage loss: {recommendations['voltage_loss']:.4f}")
print(f"  Rate loss: {recommendations['rate_loss']:.4f}")
print(f"\nCurrent ratios to CE:")
print(f"  Voltage: {recommendations['voltage_ratio']*100:.1f}%")
print(f"  Rate: {recommendations['rate_ratio']*100:.1f}%")
print(f"\nRecommended weights (for ~15% ratio):")
print(f"  voltage_weight: {recommendations['voltage_weight_recommendation']:.4f}")
print(f"  rate_weight: {recommendations['rate_weight_recommendation']:.4f}")

## 3. Precise Gradient Calibration

Backwards each loss separately and match gradient norms.

In [ ]:
# Run gradient calibration
calibrated_weights = calibrate_regularization_weights(
    model=model,
    train_loader=loader,
    loss_fn=loss_fn,
    device=device,
    num_batches=5,
    target_ratio=0.1,  # Reg grad norms = 10% of CE grad norm
    loss_components=['voltage', 'rate']
)

print("Calibrated weights:")
for comp, weight in calibrated_weights.items():
    print(f"  {comp}_weight: {weight:.4f}")

## 4. Verify Calibrated Weights

In [ ]:
# Create new loss with calibrated weights
calibrated_loss_fn = CombinedLoss(
    LossConfig(
        ce_weight=1.0,
        voltage_weight=calibrated_weights.get('voltage', 0.01),
        rate_weight=calibrated_weights.get('rate', 0.1),
    ),
    dt=1.0
)

# Check again
recommendations2 = quick_calibration_check(model, loader, calibrated_loss_fn, device, num_batches=3)

print("After calibration:")
print(f"  CE loss: {recommendations2['ce_loss']:.4f}")
print(f"  Voltage loss: {recommendations2['voltage_loss']:.4f}")
print(f"  Rate loss: {recommendations2['rate_loss']:.4f}")
print(f"\nRatios to CE:")
print(f"  Voltage: {recommendations2['voltage_ratio']*100:.1f}%")
print(f"  Rate: {recommendations2['rate_ratio']*100:.1f}%")

## 5. Compare Training Dynamics

In [ ]:
def train_step(model, x, target, optimizer, loss_fn):
    x, target = x.to(device), target.to(device)
    functional.reset_net(model.rnn, batch_size=x.shape[1])
    
    optimizer.zero_grad()
    
    with environ.context(dt=1.0):
        output, states = model(x)
    
    T = x.shape[0]
    loss, loss_dict = loss_fn(output, target, states, T)
    loss.backward()
    
    # Measure gradient norms
    total_grad_norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
    
    optimizer.step()
    
    return loss.item(), loss_dict, total_grad_norm

# Compare uncalibrated vs calibrated
results = {'uncalibrated': [], 'calibrated': []}

for name, lf in [('uncalibrated', loss_fn), ('calibrated', calibrated_loss_fn)]:
    model_copy = SingleLayerGLIFRSNN(80, 10, 128, 64).to(device)
    functional.init_net_state(model_copy.rnn, batch_size=32, device=device)
    uniform_v_(model_copy.neuron, set_reset_value=True)
    optimizer = torch.optim.Adam(model_copy.parameters(), lr=1e-3)
    
    for i, (x, target) in enumerate(loader):
        if i >= 20:
            break
        loss, loss_dict, grad_norm = train_step(model_copy, x, target, optimizer, lf)
        results[name].append({'loss': loss, 'grad_norm': grad_norm, **loss_dict})

print("Training comparison (20 steps):")
for name in ['uncalibrated', 'calibrated']:
    avg_loss = sum(r['loss'] for r in results[name]) / len(results[name])
    avg_grad = sum(r['grad_norm'] for r in results[name]) / len(results[name])
    print(f"\n{name}:")
    print(f"  Avg loss: {avg_loss:.4f}")
    print(f"  Avg grad norm: {avg_grad:.4f}")

**Key Takeaways:**

1. **Quick check** - Monitor loss component ratios (target ~10-20% of CE)
2. **Precise calibration** - Match gradient norms for exact control
3. **Voltage + rate balance** - These can conflict (high voltage promotes spikes, rate penalizes them)
4. **Retune periodically** - Optimal weights may change during training